# 6장 차트 다듬기 I: 색·축·눈금·주석

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 6.1 다듬기 전 진단

In [ ]:
# C01 | 월별 표: 5장 월별 표에 영업이익 열을 더하고 지점별로 나눕니다.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

monthly = pd.DataFrame({
    'month': [1, 2, 3, 4] * 2,
    'branch': ['A'] * 4 + ['B'] * 4,
    'sales': [100, 120, 150, 130, 70, 90, 100, 120],   # 매출(백만원)
    'profit': [10, 15, 25, 20, 7, 9, 12, 18],          # 영업이익(백만원)
})
target = 120                                        # 가정한 월 매출 목표
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
a = monthly[monthly['branch'] == 'A']              # A지점 행만
b = monthly[monthly['branch'] == 'B']              # B지점 행만
print(monthly)

In [ ]:
# C02 | 수정 전·후: 같은 값을 두 번 그리고 오른쪽만 다듬습니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), sharey=True)
for ax in axes:
    ax.plot(a['month'], a['sales'], marker='o', label='A')
    ax.plot(b['month'], b['sales'], marker='s', label='B')
    ax.set(xticks=[1, 2, 3, 4], ylim=(0, 180), xlabel='월')
axes[0].set(title='기본 차트', ylabel='매출 (백만원)')
axes[0].legend()
axes[1].lines[0].set(color=branch_colors['A'], linewidth=2.8)   # A지점 선
axes[1].lines[1].set(color='#888888', linestyle='--')          # B지점 선
peak = a.loc[a['sales'].idxmax()]
axes[1].annotate(f"{peak['month']}월 최고 {peak['sales']}",
                 xy=(peak['month'], peak['sales']), xytext=(1.4, 160),
                 arrowprops={'arrowstyle': '->'})
axes[1].set_title('A지점의 3월 최고 매출')
axes[1].legend(frameon=False, loc='lower right')
fig.tight_layout()
plt.show()

## 6.2 색·선·마커

In [ ]:
# C03 | 색상 사전: 지점마다 정한 색과 선·마커를 차트가 바뀌어도 씁니다.
fig, ax = plt.subplots(figsize=(6, 2.6))
for branch, part in monthly.groupby('branch'):   # 지점 이름과 그 행
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's', label=branch)
ax.set(title='지점별 월 매출', xlabel='월', ylabel='매출 (백만원)',
       xticks=[1, 2, 3, 4], ylim=(0, 170))
ax.legend(title='지점', frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# C04 | 값을 색으로: 크기는 순차형, 목표 대비는 발산형 팔레트로 칠합니다.
table = monthly.pivot(index='branch', columns='month', values='sales')
fig, axes = plt.subplots(1, 2, figsize=(6, 2.2))
sns.heatmap(table, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, linecolor='#999999', ax=axes[0])   # 칸 테두리
sns.heatmap(table - target, annot=True, fmt='.0f', cmap='RdBu_r',
            center=0, linewidths=0.5, linecolor='#999999',
            ax=axes[1])                  # 0을 가운데 색으로
axes[0].set(title='매출 (백만원)', xlabel='월', ylabel='지점')
axes[1].set(title='목표 대비 (백만원)', xlabel='월', ylabel='')
for ax in axes:
    ax.tick_params(axis='y', rotation=0)   # 세워 둔 라벨을 눕힘
fig.tight_layout()
plt.show()

### 직접 해 보기

**W6-01** C03을 영업이익 차트로 바꾸고 이번에는 B지점을 강조합니다.

1. C03을 복사해 `sales`를 `profit`으로, Y축 라벨·제목·범위(`ylim`, 0~30)를 영업이익에 맞게 바꿉니다.
2. A지점은 회색(`'#888888'`)으로, B지점은 사전의 색·파선·사각형을 굵게(`linewidth`) 그립니다. 조건식을 써도 되고 C02처럼 `legend()` 앞에서 `ax.lines[0].set()`으로 바꿔도 됩니다.
3. B지점 영업이익의 1~4월 변화를 숫자와 단위로 한 문장에 씁니다.

**완료 기준**: 색상 사전을 썼고, B지점은 지점 색·파선·사각형, A지점은 회색 실선·원이며, 범례가 그림과 같고, B지점의 변화를 단위와 함께 적었다.

In [ ]:
# W6-01 — 여기에서 직접 해 봅니다.

## 6.3 축·눈금·단위

In [ ]:
# C05 | 축 범위: 같은 값을 두 범위로 그려 인상을 비교합니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.1))
for ax in axes:
    ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
    ax.set(xticks=[1, 2, 3, 4], xlabel='월', ylabel='매출 (백만원)')
axes[0].set(title='전체 규모: 0~170', ylim=(0, 170))
axes[1].set(title='변화 확대: 95~155', ylim=(95, 155))
fig.tight_layout()
plt.show()

In [ ]:
# C06 | 로그 축: 같은 비율의 변화를 같은 높이 차이로 그립니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6))
for ax in axes:
    ax.plot(a['month'], a['sales'], marker='o',
            color=branch_colors['A'], label='매출')
    ax.plot(a['month'], a['profit'], marker='^', linestyle=':',
            color=branch_colors['A'], label='영업이익')
    ax.set(xticks=[1, 2, 3, 4], xlabel='월', ylabel='A지점 금액 (백만원)')
axes[0].set(title='선형 축', ylim=(0, 180))
axes[0].legend(frameon=False, loc='upper left')
axes[1].set_yscale('log')                          # Y축을 로그 축으로
axes[1].set_yticks([10, 20, 50, 100, 200],
                   labels=['10', '20', '50', '100', '200'])
axes[1].set(title='로그 축', ylim=(8, 200))
fig.tight_layout()
plt.show()

In [ ]:
# C07 | 눈금 위치와 표기: 위치는 Locator, 라벨은 Formatter가 정합니다.
from matplotlib.ticker import MultipleLocator, MaxNLocator, FuncFormatter
margin = a['profit'] / a['sales']    # 영업이익률(비율: 0.10이 10%)
month_fmt = FuncFormatter(lambda x, pos: f'{x:.0f}월')
pct_fmt = FuncFormatter(lambda y, pos: f'{y:.0%}')
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4))
axes[0].plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
axes[0].yaxis.set_major_locator(MaxNLocator(nbins=4))   # 구간 4개 이하
axes[0].set(ylabel='매출 (백만원)', ylim=(0, 180))
axes[1].plot(a['month'], margin, marker='^', linestyle=':',
             color=branch_colors['A'])          # A지점의 다른 지표(C06)
axes[1].yaxis.set_major_locator(MultipleLocator(0.05))  # 0.05마다
axes[1].yaxis.set_major_formatter(pct_fmt)
axes[1].set(ylabel='영업이익률', ylim=(0, 0.22))
for ax in axes:                                   # 두 패널의 X축을 함께
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_major_formatter(month_fmt)
fig.tight_layout()
print('계산된 Y축 눈금:', axes[0].get_yticks())
print('1월 영업이익률:', f'{margin.iloc[0]:.0%}')
plt.show()

### 직접 해 보기

**W6-02** 원자료는 그대로 두고 A지점 매출의 Y축 표기만 억원으로 바꿔 봅니다.

1. A지점 매출을 그리고 월 눈금을 `MultipleLocator(1)`로 둡니다.
2. Y축은 0~180으로 두고 눈금을 50백만원 간격으로 놓은 뒤, `FuncFormatter`에서 값을 100으로 나눠 소수 첫째 자리까지 적습니다.
3. Y축 라벨을 `매출 (억원)`으로 바꿉니다. 원자료 `a`의 월과 매출을 출력해 네 달 매출이 C01과 같은지 봅니다.

**완료 기준**: `MultipleLocator`로 눈금 간격을 정했고, 눈금 라벨이 100으로 나눈 값이며 축 라벨의 단위가 억원이고, 원자료의 네 달 매출이 C01과 같다.

In [ ]:
# W6-02 — 여기에서 직접 해 봅니다.

## 6.4 주석과 기준선

In [ ]:
# C08 | 주석: 최고점을 표에서 찾아 문구와 화살표를 함께 만듭니다.
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
peak = a.loc[a['sales'].idxmax()]            # 매출이 가장 큰 행
ax.annotate(f"{peak['month']}월 최고 {peak['sales']}백만원",
            xy=(peak['month'], peak['sales']),     # 가리킬 점(데이터 좌표)
            xytext=(0.08, 0.91), textcoords='axes fraction',  # 글자 자리
            arrowprops={'arrowstyle': '->', 'color': '#333333'})
ax.text(0.02, 0.03, '자료: 교육용 가상 자료(월 합계)',   # Axes 왼쪽 아래
        transform=ax.transAxes, fontsize=9, color='#666666')
ax.set(xticks=[1, 2, 3, 4], ylim=(0, 190), xlabel='월',
       ylabel='매출 (백만원)')
fig.tight_layout()
plt.show()

In [ ]:
# C09 | 기준선: 가정한 목표를 가로선으로 긋고 범례에 가정이라고 적습니다.
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'],
        label='A지점')
ax.axhline(target, color='#888888', linestyle='--', linewidth=1.2,
           label=f'가정한 목표 {target}')
ax.set(xticks=[1, 2, 3, 4], ylim=(0, 180), xlabel='월',
       ylabel='매출 (백만원)')
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
print('목표 이상인 달:', a.loc[a['sales'] >= target, 'month'].tolist())
print('목표 초과인 달:', a.loc[a['sales'] > target, 'month'].tolist())
plt.show()

### 직접 해 보기

**W6-03** 주석이 데이터와 함께 바뀌는지 검수합니다.

1. `b2 = b.copy()`로 B지점 표의 복사본을 만들고 복사본의 2월 매출만 160으로 바꿉니다(`b2.loc[b2['month'] == 2, 'sales'] = 160`). 원자료는 그대로 둡니다.
2. 1×2 그림(`sharey=True`)의 두 패널에 `b`와 `b2`를 그립니다. 패널마다 C08처럼 그 표의 `idxmax()` 행으로 주석 문구와 `xy`를 함께 만들고, 출처는 Axes 좌표에 둡니다. 두 패널에 같은 코드를 한 번씩 써도 됩니다.
3. 주석이 선이나 점을 가리면 `xytext`를 옮깁니다.
4. 두 패널의 최고점(월, 매출)과 `b`의 2월 매출을 출력하고, 무엇이 바뀌었는지 한 문장으로 씁니다.

**완료 기준**: 두 표 각각 `idxmax()`로 최고 행을 찾았고, 주석 문구와 `xy`가 같은 행에서 나왔으며, 출처가 Axes 좌표에 있고, 원자료 `b`가 그대로임을 출력으로 확인했으며, 바뀐 결과를 한 문장으로 설명했다.

In [ ]:
# W6-03 — 여기에서 직접 해 봅니다.

## 6.5 최종 차트와 동료 검수

In [ ]:
# C10 | 최종 차트: 색·눈금·주석·구간 강조를 한 차트에 모읍니다.
from matplotlib.ticker import MultipleLocator
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.axvspan(3, 4, color='#EEEEEE')               # 3~4월 구간을 옅게 칠함
for branch, part in monthly.groupby('branch'):
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's',
            linewidth=2.4 if branch == 'A' else 1.4, label=branch)
peak = a.loc[a['sales'].idxmax()]
ax.annotate(f"A: {peak['month']}월 최고 {peak['sales']}",
            xy=(peak['month'], peak['sales']), xytext=(1.9, 168),
            arrowprops={'arrowstyle': '->', 'color': '#333333'})
ax.set(title='A지점은 3월 최고 매출 후 4월 감소', xlabel='월',
       ylabel='매출 (백만원)', xticks=[1, 2, 3, 4], ylim=(0, 185))
ax.yaxis.set_major_locator(MultipleLocator(50))
ax.text(0.02, 0.03, '자료: 교육용 가상 자료(월 합계)',
        transform=ax.transAxes, fontsize=9, color='#666666')
ax.legend(title='지점', frameon=False, loc='lower center')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
change = a['sales'].iloc[3] - a['sales'].iloc[2]   # 4월 − 3월
print('A지점 3→4월 변화:', change, '백만원')
plt.show()

### 직접 해 보기

**W6-04** A·B지점의 3→4월 매출 변화를 계산하고 비교 차트로 저장합니다.

1. 5장 C10처럼 월 × 지점 표 `wide`를 만들고 `wide.loc[4] - wide.loc[3]`(4월 행 − 3월 행)으로 두 지점의 변화를 구합니다.
2. C10을 참고해 두 지점을 지점 색으로 같은 Y축에 그리고, 제목에 두 변화를 숫자와 단위로 적습니다.
3. `ch06_output/W6-04.png`로 저장해 열고(2장 C05·C06) 잘림을 봅니다.
4. C10과 견주어 바꾼 점과 이유 한 문장, 관찰 두 문장, 한계 한 문장을 씁니다(원인은 단정하지 않음).

**완료 기준**: 계산한 두 변화가 제목의 숫자와 같고, 두 지점이 지점 색·같은 Y축이며, 저장본에 잘림이 없고, 바꾼 점·관찰·한계 문장이 있다.

In [ ]:
# W6-04 — 여기에서 직접 해 봅니다.

## 6.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W6-01

In [ ]:
# W6-01 | 풀이: C03을 영업이익 차트로 바꾸고 B지점을 강조합니다.
fig, ax = plt.subplots(figsize=(6, 2.6))
for branch, part in monthly.groupby('branch'):
    if branch == 'A':                          # 강조하지 않는 지점은 회색 실선·원
        ax.plot(part['month'], part['profit'], color='#888888',
                linestyle='-', marker='o', label=branch)
    else:                                      # B지점은 지점 색·파선·사각형을 굵게
        ax.plot(part['month'], part['profit'], color=branch_colors['B'],
                linestyle='--', marker='s', linewidth=2.8, label=branch)
ax.set(title='B지점 영업이익: 1월 7 → 4월 18백만원', xlabel='월',
       ylabel='영업이익 (백만원)', xticks=[1, 2, 3, 4], ylim=(0, 30))
ax.legend(title='지점', frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
print(b[['month', 'profit']])
print([(line.get_color(), line.get_linestyle(), line.get_marker()) for line in ax.lines])
assert ax.lines[1].get_color() == branch_colors['B'] and ax.lines[0].get_color() == '#888888'
# 답(예): B지점 영업이익은 1월 7백만원에서 4월 18백만원으로 11백만원 늘었다.

### W6-02

In [ ]:
# W6-02 | 풀이: 원자료는 그대로 두고 Y축 표기만 억원으로 바꿉니다.
from matplotlib.ticker import MultipleLocator, FuncFormatter

fig, ax = plt.subplots(figsize=(6, 2.4))
ax.plot(a['month'], a['sales'], marker='o', color=branch_colors['A'])
ax.xaxis.set_major_locator(MultipleLocator(1))             # 1. 월 눈금
ax.yaxis.set_major_locator(MultipleLocator(50))            # 2. 50백만원 간격
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f'{y / 100:.1f}'))   # 100으로 나눠 억원
ax.set(title='A지점 1~4월 매출', xlabel='월', ylabel='매출 (억원)', ylim=(0, 180))   # 3. 축 라벨의 단위
fig.tight_layout()
fig.canvas.draw()                                          # 눈금 라벨을 계산해 둡니다
print('Y 눈금 라벨:', [t.get_text() for t in ax.get_yticklabels()])
print(a[['month', 'sales']])                               # 원자료 네 달(C01과 대조)
plt.show()
assert a['sales'].tolist() == [100, 120, 150, 130] and ax.get_ylabel() == '매출 (억원)'

### W6-03

In [ ]:
# W6-03 | 풀이: 복사본의 2월만 바꿔 주석이 데이터와 함께 바뀌는지 봅니다.
b2 = b.copy()
b2.loc[b2['month'] == 2, 'sales'] = 160            # 복사본만 바꿈(원자료 b는 그대로)

fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), sharey=True)
peak = b.loc[b['sales'].idxmax()]                  # 원자료의 최고 행
axes[0].plot(b['month'], b['sales'], marker='s', linestyle='--', color=branch_colors['B'])
axes[0].annotate(f"{peak['month']}월 최고 {peak['sales']}",
                 xy=(peak['month'], peak['sales']), xytext=(1.2, 168),
                 arrowprops={'arrowstyle': '->', 'color': '#333333'})
axes[0].text(0.02, 0.03, '자료: 교육용 가상 자료(월 합계)',
             transform=axes[0].transAxes, fontsize=9, color='#666666')
axes[0].set(title='원자료 b', ylabel='매출 (백만원)')

peak2 = b2.loc[b2['sales'].idxmax()]               # 복사본의 최고 행
axes[1].plot(b2['month'], b2['sales'], marker='s', linestyle='--', color=branch_colors['B'])
axes[1].annotate(f"{peak2['month']}월 최고 {peak2['sales']}",
                 xy=(peak2['month'], peak2['sales']), xytext=(2.6, 175),
                 arrowprops={'arrowstyle': '->', 'color': '#333333'})
axes[1].text(0.02, 0.03, '자료: 교육용 가상 자료(월 합계, 2월 고침)',
             transform=axes[1].transAxes, fontsize=9, color='#666666')
axes[1].set(title='복사본 b2')
for ax in axes:
    ax.set(xticks=[1, 2, 3, 4], ylim=(0, 190), xlabel='월')
fig.tight_layout()
plt.show()

In [ ]:
print('b의 최고점:', peak['month'], '월', peak['sales'], '| b2의 최고점:', peak2['month'], '월', peak2['sales'])
print('원자료 b의 2월 매출:', b.loc[b['month'] == 2, 'sales'].iloc[0])
assert b.loc[b['month'] == 2, 'sales'].iloc[0] == 90
# 답(예): 2월을 160으로 바꾼 복사본에서는 최고점이 4월 120에서 2월 160으로 옮겨 가고, 주석도 같은 행에서 함께 바뀌었다.

### W6-04

In [ ]:
# W6-04 | 풀이: 두 지점의 3→4월 매출 변화를 계산해 비교 차트로 저장합니다.
from pathlib import Path
from matplotlib.ticker import MultipleLocator

wide = monthly.pivot(index='month', columns='branch', values='sales')
change = wide.loc[4] - wide.loc[3]                 # 4월 행 − 3월 행
print(change)

output_dir = Path('ch06_output')
output_dir.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.axvspan(3, 4, color='#EEEEEE')
for branch, part in monthly.groupby('branch'):
    ax.plot(part['month'], part['sales'], color=branch_colors[branch],
            linestyle='-' if branch == 'A' else '--',
            marker='o' if branch == 'A' else 's', linewidth=2.4, label=branch)
ax.set(title=f"3→4월 매출 변화: A {change['A']:+d}, B {change['B']:+d}백만원",
       xlabel='월', ylabel='매출 (백만원)', xticks=[1, 2, 3, 4], ylim=(0, 185))
ax.yaxis.set_major_locator(MultipleLocator(50))
ax.text(0.02, 0.03, '자료: 교육용 가상 자료(월 합계)',
        transform=ax.transAxes, fontsize=9, color='#666666')
ax.legend(title='지점', frameon=False, loc='lower center')
fig.tight_layout()
fig.savefig(output_dir / 'W6-04.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from IPython.display import Image, display

display(Image(filename='ch06_output/W6-04.png', width=600))
assert (change['A'], change['B']) == (-20, 20)
# 바꾼 점(예): 제목을 A지점 최고점 대신 두 지점의 3→4월 변화로 바꾸고 두 선을 같은 굵기로 했다 — 비교가 이 차트의 질문이라서다.
# 관찰(예): 3월에서 4월까지 A지점 매출은 150에서 130백만원으로 20 줄었다.
# 관찰(예): 같은 기간 B지점은 100에서 120백만원으로 20 늘어 두 지점의 차이가 50에서 10백만원으로 좁혀졌다.
# 한계(예): 네 달 자료라 이 변화가 계절 때문인지 지점 사정 때문인지는 알 수 없다.